# auditPaper_assist — dense 임베딩 + Qdrant 적재 (Colab GPU)

저장소 `main`에서 입력을 직접 확보해 **BAAI/bge-m3** dense를 계산하고, 그대로 `--stage upsert`까지 수행한다.

- 사전 조건: 로컬의 코퍼스 수정이 **push 완료**된 상태여야 함 (셀 2가 신선도를 assert)
- 런타임 유형: **GPU** (T4/L4)
- 마지막 셀이 `colab_artifacts.zip`을 Google Drive `auditPaper_assist/`에 저장 → 로컬 `index/`에 풀어 배치할 것 (manifest·build_report·embed_meta는 커밋, embeddings.npy·cids.json은 gitignore 캐시)

In [7]:
# 셀 1 — GPU 확인 + 의존성 설치
!nvidia-smi -L
!pip -q install FlagEmbedding qdrant-client kiwipiepy

GPU 0: NVIDIA L4 (UUID: GPU-4966008b-4b13-bb83-7aff-c7de7c4f4a67)


In [8]:
# 셀 2 — 저장소 확보 + 코퍼스 커밋 SHA 기록 + 입력 신선도 검증
import json, os, urllib.request

REPO = "worldoftoddl/auditPaper_MCP"  # 2026-07 저장소 개명 반영
sha = json.load(urllib.request.urlopen(f"https://api.github.com/repos/{REPO}/commits/main"))["sha"]
os.environ["CORPUS_COMMIT"] = sha
print("코퍼스 커밋:", sha)

!wget -q https://codeload.github.com/{REPO}/tar.gz/refs/heads/main -O repo.tgz
!rm -rf repo && mkdir -p repo && tar xzf repo.tgz -C repo --strip-components=1
assert os.path.isfile("repo/scripts/build_index.py"), "저장소 확보 실패"

rows = [json.loads(l) for l in open("repo/index/embed_input.jsonl", encoding="utf-8")]
EXPECTED = 20552   # 정본 10,062 + BC 8,721 + IE 1,753 + 분할 순증 16 (v2 확장, 2026-07-12)
assert len(rows) == EXPECTED, f"행 수 {len(rows)} != {EXPECTED}"
# 신선도: BC·IE 갈래 확장(v2)이 반영된 입력인지 확인 — 미반영이면 push 후 재실행
bc1 = next((r for r in rows if r["cid"] == "KIFRS::1116::BC1"), None)
assert bc1 is not None, "구판 입력(BC·IE 갈래 부재) — push 후 재실행"
assert any(r["cid"] == "KIFRS::1115::IE3" for r in rows), "IE 갈래 부재 — push 후 재실행"
print("입력 확보·신선도 OK:", len(rows), "행 (BC·IE 갈래 포함)")

코퍼스 커밋: 9f6686497fb699a0327ad800efe77f5055ddfd92
입력 확보·신선도 OK: 20552 행 (BC·IE 갈래 포함)


In [9]:
# 셀 3 — bge-m3 dense 인코딩 (T4 기준 5~10분) + 정규화 보증
import numpy as np, torch
from FlagEmbedding import BGEM3FlagModel

assert torch.cuda.is_available(), "GPU 런타임 아님"
MODEL = "BAAI/bge-m3"
model = BGEM3FlagModel(MODEL, use_fp16=True)
out = model.encode([r["text"] for r in rows], batch_size=64, max_length=8192,
                   return_dense=True, return_sparse=False, return_colbert_vecs=False)
emb = np.asarray(out["dense_vecs"], dtype="float32")
assert emb.shape == (EXPECTED, 1024), emb.shape
assert not np.isnan(emb).any(), "NaN 존재"
n = np.linalg.norm(emb, axis=1)
print("norm range:", n.min(), n.max())
if not np.allclose(n, 1.0, atol=1e-3):
    print("[경고] 노름 이탈 → 재정규화")
    emb = emb / n[:, None]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 429/429 [01:56<00:00,  3.69it/s]


norm range: 0.9996698 1.0003407


In [12]:
# 셀 4 — 산출물 + 산출물 매니페스트(embed_meta.json) 저장 → repo/index/ 배치
import hashlib, datetime
from importlib.metadata import version as pkg_ver  # FlagEmbedding은 __version__ 미노출

def ver(name):
    try:
        return pkg_ver(name)
    except Exception:
        return "unknown"

np.save("embeddings.npy", emb)
cids = [r["cid"] for r in rows]
json.dump(cids, open("cids.json", "w", encoding="utf-8"), ensure_ascii=False)
!cp embeddings.npy cids.json repo/index/   # 메타 생성이 실패해도 본 산출물은 배치되도록 선복사

sha256 = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
meta = {
    "embedding_device": f"colab-{torch.cuda.get_device_name(0)}-fp16",
    "embedding_runtime": (f"FlagEmbedding {ver('FlagEmbedding')}, "
                          f"torch {ver('torch')}, transformers {ver('transformers')}"),
    "model": MODEL,
    "rows": len(rows), "dim": int(emb.shape[1]), "dtype": str(emb.dtype),
    "corpus_commit": os.environ["CORPUS_COMMIT"],
    "input_sha256": sha256("repo/index/embed_input.jsonl"),
    "embeddings_sha256": sha256("embeddings.npy"),
    "cids_sha256": sha256("cids.json"),
    "generated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
json.dump(meta, open("embed_meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
!cp embed_meta.json repo/index/
print(json.dumps(meta, ensure_ascii=False, indent=2))

{
  "embedding_device": "colab-NVIDIA L4-fp16",
  "embedding_runtime": "FlagEmbedding 1.4.0, torch 2.11.0+cu128, transformers 5.12.1",
  "model": "BAAI/bge-m3",
  "rows": 20552,
  "dim": 1024,
  "dtype": "float32",
  "corpus_commit": "9f6686497fb699a0327ad800efe77f5055ddfd92",
  "input_sha256": "cc1caa1ea14142a5be04347d2a5a089e99f8e9501ca2f3d8c102f73d15f15c38",
  "embeddings_sha256": "4557cf8cb88c219a10f70c5a43d979de4fb1ad9000504991c43b3646b9454a54",
  "cids_sha256": "f1a20639e890b9e6d5b0bb5743041f6f7a57dd901826ba426e61d1d15f4a0101",
  "generated_at": "2026-07-12T07:59:28.963303+00:00"
}


In [13]:
# 셀 5 — Qdrant 접속 정보 입력 + 업서트·점검·스모크·매니페스트 (전 과정)
for f in ("embeddings.npy", "cids.json", "embed_meta.json"):
    assert os.path.isfile(f"repo/index/{f}"), f"repo/index/{f} 없음 — 셀 4부터 재실행"

from getpass import getpass
os.environ["QDRANT_URL"] = getpass("QDRANT_URL: ").rstrip("/")
os.environ["QDRANT_API_KEY"] = getpass("QDRANT_API_KEY: ")

%cd /content/repo
!python scripts/build_index.py --stage upsert --embeddings index/embeddings.npy --cids index/cids.json
%cd /content

/content/repo
[F] 용어 사전 파생...
[E] sparse 벡터 (kiwipiepy + BM25 문서측 가중)...
  sparse 토큰화 2000/20552
  sparse 토큰화 4000/20552
  sparse 토큰화 6000/20552
  sparse 토큰화 8000/20552
  sparse 토큰화 10000/20552
  sparse 토큰화 12000/20552
  sparse 토큰화 14000/20552
  sparse 토큰화 16000/20552
  sparse 토큰화 18000/20552
  sparse 토큰화 20000/20552
[G] Qdrant 적재: standards_20250829_bgem3_v2
  업서트 256/20552
  업서트 512/20552
  업서트 768/20552
  업서트 1024/20552
  업서트 1280/20552
  업서트 1536/20552
  업서트 1792/20552
  업서트 2048/20552
  업서트 2304/20552
  업서트 2560/20552
  업서트 2816/20552
  업서트 3072/20552
  업서트 3328/20552
  업서트 3584/20552
  업서트 3840/20552
  업서트 4096/20552
  업서트 4352/20552
  업서트 4608/20552
  업서트 4864/20552
  업서트 5120/20552
  업서트 5376/20552
  업서트 5632/20552
  업서트 5888/20552
  업서트 6144/20552
  업서트 6400/20552
  업서트 6656/20552
  업서트 6912/20552
  업서트 7168/20552
  업서트 7424/20552
  업서트 7680/20552
  업서트 7936/20552
  업서트 8192/20552
  업서트 8448/20552
  업서트 8704/20552
  업서트 8960/20552
  업서트 9216/20552
  업서트 9472/20552
  업서트 9728/2

In [14]:
# 셀 6 — 산출물 회수: Google Drive에 zip으로 저장
# 실행하면 Drive 권한 승인 팝업이 뜸 → 승인 후 내 드라이브의 auditPaper_assist/ 폴더에 저장됨
import zipfile, os
from google.colab import drive
drive.mount("/content/drive")

with zipfile.ZipFile("colab_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("repo/index/manifest.json", "manifest.json")
    z.write("repo/index/build_report.md", "build_report.md")
    z.write("embed_meta.json", "embed_meta.json")
    z.write("embeddings.npy", "embeddings.npy")
    z.write("cids.json", "cids.json")

DEST = "/content/drive/MyDrive/auditPaper_assist"
os.makedirs(DEST, exist_ok=True)
!cp colab_artifacts.zip "$DEST"/
!ls -lh "$DEST"/colab_artifacts.zip
print("저장 완료 → 내 드라이브 > auditPaper_assist > colab_artifacts.zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw------- 1 root root 48M Jul 12 08:06 /content/drive/MyDrive/auditPaper_assist/colab_artifacts.zip
저장 완료 → 내 드라이브 > auditPaper_assist > colab_artifacts.zip
